# Bank Marketing Subscription Prediction
End-to-end classification project focused on realistic CRM deployment and data leakage prevention.

This notebook covers **EDA and model evaluation only**.
Training logic lives in `src/` — imported here to keep concerns separated.

## Imports

In [ ]:
import sys
sys.path.append('..')  # make src/ importable from notebooks/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap

from sklearn.metrics import confusion_matrix, roc_auc_score

from src.train import load_data, split_features, make_splits, train, save_model

import warnings
warnings.filterwarnings('ignore')

## Load Data

In [ ]:
df = load_data('../data/bank.csv')
df.head()

In [ ]:
df.info()

In [ ]:
print(f'Shape: {df.shape}')
print(f'Missing values:\n{df.isnull().sum()}')

The dataset contains 11,162 observations and 17 features. No missing values detected.

## Exploratory Data Analysis
### Target Distribution
Understanding class imbalance before modelling.

In [ ]:
print(df['deposit'].value_counts(normalize=True).round(3) * 100)

sns.countplot(x='deposit', data=df)
plt.title('Target Distribution')
plt.show()

Moderate class imbalance: more non-subscribers than subscribers.
Accuracy alone is insufficient — ROC-AUC, precision, recall are primary metrics.

### Numerical Features
Distribution and outlier analysis.

In [ ]:
df[['age', 'balance', 'duration', 'campaign']].hist(figsize=(10, 8))
plt.suptitle('Numerical Feature Distributions')
plt.tight_layout()
plt.show()

- **Age**: well-distributed, 30–50 range dominant.
- **Balance**: highly right-skewed → log1p transformation applied in pipeline.
- **Duration**: strong skew → data leakage risk (see below).
- **Campaign**: concentrated at low values.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col in zip(axes, ['balance', 'duration', 'campaign']):
    sns.boxplot(y=df[col], ax=ax)
    ax.set_title(col.capitalize())
plt.suptitle('Outlier Analysis')
plt.tight_layout()
plt.show()

Outliers present in balance and duration. Not removed — tree-based models are robust to skewness and outliers.

### Categorical Features vs Target

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

sns.countplot(x='job', hue='deposit', data=df, ax=axes[0])
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].set_title('Job vs Deposit (Count)')

pd.crosstab(df['job'], df['deposit'], normalize='index').plot(
    kind='bar', stacked=True, ax=axes[1]
)
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=45, ha='right')
axes[1].set_title('Job vs Deposit (Proportion)')

plt.tight_layout()
plt.show()

Students and retired customers show higher subscription rates. Blue-collar and services workers show lower engagement.
Occupation is a strong predictive feature.

### Data Leakage: `duration`

`duration` = length of the last call. Available **only after** the call ends — not at prediction time.

Strategy:
- **With duration**: benchmark only (not deployable).
- **Without duration**: realistic deployment scenario.

## Training
### Scenario 1 — With Duration (benchmark)

In [ ]:
X_wd, y_wd = split_features(df, 'with_duration')
X_train_wd, X_val_wd, X_test_wd, y_train_wd, y_val_wd, y_test_wd = make_splits(X_wd, y_wd)

pipeline_wd, threshold_wd, model_name_wd, metrics_wd, _ = train(
    X_train_wd, X_val_wd, X_test_wd, y_train_wd, y_val_wd, y_test_wd, 'with_duration'
)

In [ ]:
y_proba_wd = pipeline_wd.predict_proba(X_test_wd)[:, 1]
y_pred_wd = (y_proba_wd >= threshold_wd).astype(int)

cm = confusion_matrix(y_test_wd, y_pred_wd)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Subscribe', 'Subscribe'],
            yticklabels=['Not Subscribe', 'Subscribe'])
plt.title('Confusion Matrix — With Duration (benchmark, data leakage)')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

### Scenario 2 — Without Duration (realistic deployment)

In [ ]:
X_nd, y_nd = split_features(df, 'without_duration')
X_train_nd, X_val_nd, X_test_nd, y_train_nd, y_val_nd, y_test_nd = make_splits(X_nd, y_nd)

pipeline_nd, threshold_nd, model_name_nd, metrics_nd, _ = train(
    X_train_nd, X_val_nd, X_test_nd, y_train_nd, y_val_nd, y_test_nd, 'without_duration'
)

In [ ]:
y_proba_nd = pipeline_nd.predict_proba(X_test_nd)[:, 1]
y_pred_nd = (y_proba_nd >= threshold_nd).astype(int)

cm = confusion_matrix(y_test_nd, y_pred_nd)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=['Not Subscribe', 'Subscribe'],
            yticklabels=['Not Subscribe', 'Subscribe'])
plt.title('Confusion Matrix — Without Duration (realistic)')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.tight_layout()
plt.show()

### Scenario Comparison

In [ ]:
comparison = pd.DataFrame({
    'Metric': ['Val ROC-AUC RF', 'CV ROC-AUC (mean)', 'Test ROC-AUC', 'Threshold'],
    'With Duration': [
        round(metrics_wd['val_roc_auc_rf'], 3),
        round(metrics_wd['cv_roc_auc_mean'], 3),
        round(metrics_wd['test_roc_auc'], 3),
        round(metrics_wd['threshold'], 3),
    ],
    'Without Duration': [
        round(metrics_nd['val_roc_auc_rf'], 3),
        round(metrics_nd['cv_roc_auc_mean'], 3),
        round(metrics_nd['test_roc_auc'], 3),
        round(metrics_nd['threshold'], 3),
    ]
})
comparison.set_index('Metric')

Performance drop without `duration` is expected and confirms the leakage hypothesis.
The realistic model (without duration) is the only one valid for deployment.

## Feature Importance — SHAP
Applied to realistic model (without duration).

In [ ]:
model = pipeline_nd.named_steps['model']
preprocessor = pipeline_nd.named_steps['preprocessor']

X_test_transformed = preprocessor.transform(X_test_nd)
feature_names = preprocessor.named_steps['column'].get_feature_names_out()

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test_transformed)
shap_values_pos = shap_values[:, :, 1]

In [ ]:
shap.summary_plot(shap_values_pos, X_test_transformed, feature_names=feature_names)

Each point = one prediction. X-axis = SHAP value (impact on output). Color = feature value magnitude.

Positive SHAP → increases probability of subscription. Negative → decreases.

In [ ]:
shap.summary_plot(shap_values_pos, X_test_transformed, feature_names=feature_names, plot_type='bar')

In [ ]:
shap_importance = np.abs(shap_values_pos).mean(axis=0)
shap_df = pd.DataFrame({'feature': feature_names, 'shap_importance': shap_importance})
shap_df.sort_values('shap_importance', ascending=False).head(10)

**Top drivers:**
1. `poutcome_success` — previous campaign success strongly predicts subscription.
2. `contact_unknown` / `contact_cellular` — communication channel matters.
3. `housing_yes` — financial situation signal.
4. `pdays` — recency of previous contact.
5. `balance` — account balance.

Model relies on **behavioral and historical** features, not purely demographic.

## Conclusions

- **Best model**: Random Forest (tuned via GridSearchCV), selected in both scenarios.
- **Realistic ROC-AUC**: 0.773 without duration — acceptable for deployment.
- **Threshold**: tuned on validation set to enforce recall ≥ 0.75 for class 1.
- **SHAP**: previous campaign outcome and contact channel are dominant predictors.

**MLOps pipeline**: training logic in `src/train.py`, served via FastAPI (`api/`), tracked with MLflow.